Runnable = the fundamental building block  
Chain = composition of runnables  

# Runnables
- You can consider runnables as a unit of work.
- Any component that can take input → process it → return output is considered as runnable
- Each runnable in langchain has a specific task.
- Each runnable has a commont interface like
    1. invoke() - Single input → single output
    2. batch() - Multiple inputs → multiple outputs
    3. stream() - Returns output token-by-token
    4. ainvoke() - Used in production systems (Async)
- Each runnable can be connectable, So we can execute highly complex workflow

### Types of Runnables in Langchain:
1. Task Specific Runnables:
    - Core langchain components that have been converted into runnables so they can be used in pipelines.
    - Purpose: Perform task-specific operations like LLM Calls, prompting, retrieval, etc
    - Example:
        1. ChatOpenAI - Runs an LLM model
        2. PromptTemplate - Formats prompts dynamically.
        3. Retriever - Retrieves relevant documents.

2. Runnable Primitives:
    - Fundamentals building blocks for structuring execution logic in AI workflows.
    - Purpose: They help orchestrate execution by defining how different runnables interact i.e sequentially, parallel, conditional, etc.
    - Example:
        1. RunnableSequence - Runs steps in order ( | )
        2. RunnableParallel - Runs multiple steps simultaneously
        3. RunnableMap - Maps the same input accross multiple parts
        4. RunnableBranch - Implement Conditional execution
        5. RunnableLambda - Wraps custom python functions into runnables
        6. RunnablePassthrough - Just forwards input as output (acts as a placeholder)

### Runnables Primitives

In [ ]:
# 1. RunnableSequence: It is a class that allows you to chain together multiple runnables (like prompts, models, parsers) in a sequence. Each runnable's output is passed as input to the next runnable in the sequence.
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence

load_dotenv()

prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

model = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)
parser = StrOutputParser()

prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

chain = RunnableSequence(prompt1, model, parser, prompt2, model, parser)

print(chain.invoke({'topic':'AI'}))

d:\Repos\Agents\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**What’s going on in the joke?**

1. **Hide‑and‑seek** – In the game, the “hider” must stay out of sight.  
2. **“Logging”** – In computing, to *log* something means to record it (e.g., write a line to a log file).  
3. **AI’s habit** – An AI (or any program) often keeps logs of its actions, status, or location for debugging, monitoring, or training purposes.

**Punchline:**  
When the AI tries to hide, it *keeps logging its location*—so it’s literally revealing where it is, making it impossible to stay hidden. The humor comes from the double meaning of “logging” (recording data vs. the act of hiding) and the idea that a machine that’s always tracking itself can’t play a game that relies on secrecy.

So the joke is a pun on the technical term “logging” and the literal act of hiding.


In [ ]:
# 2. RunnableParallel: It is a class that allows you to run multiple runnables in parallel. Each runnable can have its own input and output, and the results are returned in a dictionary format where the keys are the names of the runnables and the values are their respective outputs.
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel

load_dotenv()

prompt1 = PromptTemplate(
    template='Generate a tweet about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a Linkedin post about {topic}',
    input_variables=['topic']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'tweet': RunnableSequence(prompt1, model, parser),
    'linkedin': RunnableSequence(prompt2, model, parser)
})

result = parallel_chain.invoke({'topic':'AI'})

print("Result of the tweet pipeline:\n",result['tweet'], "\n")
print("-------------------------------\n")
print("Result of the LinkedIn pipeline:\n",result['linkedin'], "\n")

Result of the tweet pipeline:
 🤖✨ AI isn’t just a tool—it’s a new lens on the world. From creative art to climate modeling, it’s reshaping how we solve problems. But with great power comes responsibility: transparency, fairness, and human‑centric design must guide every line of code. #AI #FutureTech 

-------------------------------

Result of the LinkedIn pipeline:
 🚀 **AI: The Catalyst for Tomorrow’s Business Landscape** 🚀  

In the past year alone, AI has moved from “nice‑to‑have” to “must‑have” for companies that want to stay competitive. From generative models that draft marketing copy in seconds to predictive analytics that uncover hidden customer insights, the possibilities are expanding faster than we can keep up with.

**Why it matters now**

1. **Speed & Scale** – Automate routine tasks, freeing up teams to focus on strategy and creativity.  
2. **Personalization at Scale** – Deliver hyper‑relevant experiences that drive loyalty and revenue.  
3. **Data‑Driven Decision Making

In [ ]:
# 3. RunnablePassthrough: It is a class that allows you to pass the input directly to the output without any modification. This can be useful in scenarios where you want to include the original input in the output of a chain or when you want to create a placeholder for a runnable that will be defined later.
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough

load_dotenv()

prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

parser = StrOutputParser()

prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

joke_gen_chain = RunnableSequence(prompt1, model, parser)

parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'explanation': RunnableSequence(prompt2, model, parser)
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

print(final_chain.invoke({'topic':'cricket'}))

{'joke': 'Why did the cricket team go to the bakery?  \nBecause they heard the batter was on a roll!', 'explanation': '**The joke is a pun that mixes two very different meanings of the word “batter” and the idiom “on a roll.”**\n\n| Element | Cricket meaning | Baking meaning | Idiomatic meaning |\n|---------|-----------------|----------------|-------------------|\n| **Batter** | The player who is batting (i.e., the “batter” of the team). | A liquid mixture of flour, eggs, milk, etc. used to make dough or cake. | A noun that can be used in the phrase “on a roll.” |\n| **On a roll** | A phrase meaning “doing very well” (e.g., a player is on a winning streak). | Literally a roll of dough or a bread roll. | The idiom “on a roll” meaning success. |\n\n**How the joke works**\n\n1. **Setup** – “Why did the cricket team go to the bakery?”  \n   The listener expects a cricket‑related answer (e.g., a player’s injury, a strategy, etc.).\n\n2. **Punchline** – “Because they heard the batter was on 

In [ ]:
# 4. RunnableLambda: It is a class that allows you to define a runnable using a lambda function. This can be useful for simple operations that do not require the overhead of creating a separate class or function.
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnablePassthrough, RunnableParallel

load_dotenv()

def word_count(text):
    return len(text.split())

prompt = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt, model, parser)

parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'word_count': RunnableLambda(word_count)
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

result = final_chain.invoke({'topic':'AI'})

final_result = """{} \n word count - {}""".format(result['joke'], result['word_count'])
print("Result", result)
print("-------------------------------\n")
print("Final Result:", final_result)

Result {'joke': 'Why did the AI refuse to play hide‑and‑seek with humans?  \n\nBecause every time it tried to hide, it kept “logging” its location!', 'word_count': 22}
-------------------------------

Final Result: Why did the AI refuse to play hide‑and‑seek with humans?  

Because every time it tried to hide, it kept “logging” its location! 
 word count - 22


In [11]:
# 5. RunnableBranch: It is a class that allows you to define a branch in the execution flow based on a condition. You can specify different runnables to execute for the true and false branches of the condition. This can be useful for creating dynamic chains that can adapt their behavior based on the input or intermediate results.
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableSequence, RunnablePassthrough, RunnableBranch

load_dotenv()

prompt1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Summarize the following text \n {text}',
    input_variables=['text']
)

parser = StrOutputParser()

report_gen_chain = prompt1 | model | parser

branch_chain = RunnableBranch(
    (lambda x: len(x.split())>300, prompt2 | model | parser),
    RunnablePassthrough()
)

final_chain = RunnableSequence(report_gen_chain, branch_chain)

print(final_chain.invoke({'topic':'Russia vs Ukraine'}))

**Russia‑Ukraine Conflict – April 2026 Summary**

| Section | Key Take‑aways |
|---------|----------------|
| **Origins** | Post‑Soviet Ukraine’s pivot to Europe strained ties with Russia. 2014 saw Crimea’s annexation and the Donbas “frozen” war, set against a backdrop of political upheaval (Euromaidan, Yanukovych’s ouster). |
| **Full‑Scale Invasion (Feb 2022)** | Russia launched a multi‑front assault, quickly gaining territory but facing stiff Ukrainian resistance. 2022‑2023 saw a series of counter‑offensives that reclaimed Kharkiv, Kherson, and pushed back Russian forces from Bakhmut and other eastern towns. Front lines remain fluid, with repeated gains and losses, especially around Kharkiv. |
| **Military Dynamics** | *Russia*: Conventional forces, heavy artillery, drones, but suffering logistical strain and high casualties. *Ukraine*: Initially under‑equipped, now bolstered by Western anti‑tank, anti‑aircraft, and HIMARS systems; relies on asymmetric tactics and volunteer units. |

In [ ]:
# 6. RunnableMap: It is a class that allows you to define a mapping of runnables that can be executed in parallel. Each runnable in the map can have its own input and output, and the results are returned in a dictionary format where the keys are the names of the runnables and the values are their respective outputs. This can be useful for creating complex chains that require multiple parallel operations on the same input.
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap, RunnablePassthrough, RunnableSequence
from dotenv import load_dotenv

load_dotenv()

report_prompt = PromptTemplate(
    template="Write a detailed report on {topic}",
    input_variables=["topic"]
)

summary_prompt = PromptTemplate(
    template="Summarize the following text:\n{text}",
    input_variables=["text"]
)

parser = StrOutputParser()

report_chain = RunnableSequence(report_prompt, model, parser)

map_chain = RunnableMap({
    "original_report": RunnablePassthrough(),
    "summary": RunnableSequence(summary_prompt, model, parser),
    "word_count": lambda x: len(x.split())
})

final_chain = RunnableSequence(report_chain, map_chain)
result = final_chain.invoke({"topic": "AI in Healthcare"})

print("Report:\n", result["original_report"])
print("\nSummary:\n", result["summary"])
print("\nWord Count:\n", result["word_count"])

Report:
 # AI in Healthcare – A Comprehensive Report

**Prepared for:** [Stakeholder / Organization]  
**Prepared by:** ChatGPT – AI Research Assistant  
**Date:** 17 April 2026  

---

## Executive Summary

Artificial Intelligence (AI) has transitioned from a nascent research curiosity to a cornerstone of modern healthcare. From predictive analytics that flag high‑risk patients to autonomous robotic surgeons, AI is reshaping every layer of the health ecosystem. This report provides a deep dive into the current state of AI in healthcare, covering:

1. **Historical evolution** and key milestones.  
2. **Core AI technologies** (machine learning, deep learning, natural language processing, reinforcement learning).  
3. **Clinical applications** – diagnostics, therapeutics, operational efficiency, and patient engagement.  
4. **Benefits** – accuracy, speed, cost‑efficiency, and scalability.  
5. **Challenges** – data quality, bias, interpretability, regulatory compliance, and workforce imp

In [13]:
print(result)

{'original_report': '# AI in Healthcare – A Comprehensive Report\n\n**Prepared for:** [Stakeholder / Organization]  \n**Prepared by:** ChatGPT – AI Research Assistant  \n**Date:** 17\u202fApril\u202f2026  \n\n---\n\n## Executive Summary\n\nArtificial Intelligence (AI) has transitioned from a nascent research curiosity to a cornerstone of modern healthcare. From predictive analytics that flag high‑risk patients to autonomous robotic surgeons, AI is reshaping every layer of the health ecosystem. This report provides a deep dive into the current state of AI in healthcare, covering:\n\n1. **Historical evolution** and key milestones.  \n2. **Core AI technologies** (machine learning, deep learning, natural language processing, reinforcement learning).  \n3. **Clinical applications** – diagnostics, therapeutics, operational efficiency, and patient engagement.  \n4. **Benefits** – accuracy, speed, cost‑efficiency, and scalability.  \n5. **Challenges** – data quality, bias, interpretability, re